## Holiday egg procurement under supply uncertainty

### What do we know?
- A grocery chain must choose a pre-order quantity for Easter-week eggs.
- Demand is uncertain and rises more often when supply conditions are tight.
- If the order is too small, the firm uses emergency procurement at a higher cost.
- If the order is too large, leftover cartons are marked down.

### Model elements
**Key output:** Profit

**Deterministic inputs**
- `c` = regular purchase cost per dozen
- `p` = selling price per dozen
- `r` = markdown / salvage value per dozen
- `Q` = pre-order quantity

**Stochastic inputs**
- supply state
- demand regime (normal or high)
- demand during the week

**Outputs**
- profit
- stockout indicator
- leftover inventory
- shortage

**For Optimal Pre-order quantity Q**
- Revenue = selling price × units sold
- Regular procurement cost = pre-order cost × Q
- Emergency procurement cost = emergency price × extra units needed beyond Q
- Salvage / markdown value = discounted recovery on leftover units
- Profit = revenue + salvage − regular cost − emergency cost
- sales = min(demand, available inventory + emergency purchases)
- leftover = max(Q − demand, 0)
- shortage = max(demand − Q, 0)

## Values:
Wholesale egg price:
$1.86 per dozen

Source:
USDA Agricultural Marketing Service Egg Market Report (March 2026).

* Retail egg price range
Source:
USDA ERS reported retail egg prices reaching $5.12 per dozen in 2025 during shortages.

* Holiday demand spikes
Source:
MarketWatch reported increased egg demand during Easter due to baking and egg-dyeing traditions.

Model Assumptions:

These are not directly observed values, but they are consistent with market evidence and economic logic.

* Retail price
Value used:
$3.49 per dozen

Retail pricing usually includes:
wholesale price + store margin + logistics + spoilage risk
Typical grocery margin on eggs is roughly 40–80% above wholesale.

Using the wholesale benchmark:
1.86 × 1.8 ≈ 3.35
So the model uses:
3.49 per dozen
which falls within the typical grocery range.
* Pre-order procurement cost
Value used:
1.90 per dozen
The market report listed:
1.86 wholesale price
Retailers typically lock in supply contracts slightly above spot wholesale due to logistics and supplier margins.
Thus the model rounds to:
1.90 per dozen.

* Emergency procurement costs
Values used
Supply state	Cost
Normal	2.30
Tight	3.10
Shortage	4.20

When supply tightens, retailers must purchase from:

• secondary wholesalers
• distributors with remaining inventory
• expedited shipments

These prices can exceed normal wholesale prices by 20–120%.

So the assumptions follow this logic:

Normal supply:
1.90 → 2.30 (~20% premium)

Tight supply:
1.90 → 3.10 (~60% premium)

Severe shortage:
1.90 → 4.20 (~120% premium)

These reflect real market spikes observed during avian flu shortages.

Salvage value
Value used

$1.20 per dozen

Why this is reasonable

Leftover eggs must be:

• discounted
• promoted in clearance
• or sold close to expiration

Typical markdowns range 30–50% off retail.

Using retail price:

3.49 × 0.35 ≈ $1.22

So the model uses:

$1.20 salvage value.

Demand distributions
Normal demand distribution

Triangular distribution

(420, 500, 590)

High demand distribution

Triangular distribution

(500, 620, 760)


How the parameters were derived:

The base demand assumption:

Average demand ≈ 500 dozen

Holiday demand increases:

10–15%

So the high demand mode becomes:

500 × 1.25 ≈ 620

Maximum demand was expanded to 760 to represent extreme demand spikes.

Supply state probabilities
Values used
State	Probability
Normal	55%
Tight	30%
Shortage	15%
Why these probabilities were chosen

The egg market has experienced:

• frequent supply disruptions
• periodic shortages
• but most weeks remain normal.

So the probabilities reflect:

• majority normal supply
• meaningful risk of disruption

which captures the real volatility described in USDA reports.

* Correlation between supply and demand
Assumption used:

Demand is more likely to spike when supply is tight.

Example in the simulation:

Supply state	Probability of high demand
Normal	25%
Tight	45%
Shortage	65%
Why this assumption is reasonable

Supply shortages often trigger:

• panic buying
• stockpiling
• higher retail traffic

This creates positive correlation between supply shocks and demand spikes.

In [ ]:
# Import libraries that may be used
import numpy as np
import pandas as pd
import scipy.stats as sp
import matplotlib.pyplot as plt

In [ ]:
def summarystats(Outp):
    '''Generic code to compute descriptive statistics for each column in a data frame Outp'''
    Mean = Outp.mean()
    StDev = Outp.std(ddof=1)
    Percent05 = Outp.quantile(0.05)
    Percent50 = Outp.quantile(0.50)
    Percent95 = Outp.quantile(0.95)
    StdErr = Outp.sem()
    Min = Outp.min()
    Max = Outp.max()

    Prob = 0.95
    Tails = (1 - Prob)/2
    z = sp.norm.ppf(1-Tails)
    CI_low = Mean - z * StdErr
    CI_high = Mean + z * StdErr

    Sample_stats = pd.concat([Mean,StDev,Percent05,Percent50,Percent95,StdErr,CI_low,CI_high,Min,Max], axis=1)
    Sample_stats.columns = ['Mean','Std Dev','5%','50%','95%','Std Error','95% CI Low','95% CI High','Min','Max']
    return Sample_stats

In [ ]:
def simulationfunction(controls,inputs):
    '''
    Generic function to structure the simulation process: we start by specifying the elements of the deterministic model,
    then we identify stochastic inputs, execute our simulation and return the outputs
    '''
    seed = controls[0]
    n = controls[1]

    c = inputs[0]
    p = inputs[1]
    r = inputs[2]
    q_grid = inputs[3]
    supply_states = inputs[4]
    supply_prob = np.array(inputs[5], dtype=float)
    emergency_cost = inputs[6]
    high_demand_prob = inputs[7]
    normal_tri = inputs[8]
    high_tri = inputs[9]

    if seed:
        np.random.seed(1)

    all_rows = []

    for q in q_grid:
        state_idx = np.random.choice(len(supply_states), size=n, p=supply_prob)
        state_names = np.array(supply_states)[state_idx]

        hi_prob = np.array([high_demand_prob[s] for s in state_names])
        is_high = np.random.rand(n) < hi_prob

        demand = np.where(
            is_high,
            np.random.triangular(high_tri[0], high_tri[1], high_tri[2], size=n),
            np.random.triangular(normal_tri[0], normal_tri[1], normal_tri[2], size=n)
        )

        Ce = np.array([emergency_cost[s] for s in state_names])

        sales = np.minimum(demand, q)
        shortage = np.maximum(demand - q, 0)
        leftover = np.maximum(q - demand, 0)
        profit = sales*p + leftover*r - q*c - shortage*Ce

        summary_row = {
            'Order Quantity': q,
            'Expected Profit': profit.mean(),
            'Std Dev Profit': profit.std(ddof=1),
            '5% Profit': pd.Series(profit).quantile(0.05),
            '50% Profit': pd.Series(profit).quantile(0.50),
            '95% Profit': pd.Series(profit).quantile(0.95),
            'Stockout Probability': (shortage > 0).mean(),
            'Average Leftover': leftover.mean(),
            'Average Shortage': shortage.mean(),
        }
        all_rows.append(summary_row)

    return pd.DataFrame(all_rows)

In [ ]:
# Simulation control parameters
controls = [True, 50000]

inputs = [
    1.86,                                  # c
    3.49,                                  # p
    1.20,                                  # r
    list(range(400, 801, 20)),             # candidate Q values
    ['normal', 'tight', 'shortage'],       # supply states
    [0.55, 0.30, 0.15],                    # supply probabilities
    {'normal': 2.30, 'tight': 3.10, 'shortage': 4.20},   # emergency cost by state
    {'normal': 0.25, 'tight': 0.45, 'shortage': 0.65},   # probability of high demand conditional on supply state
    [420, 500, 590],                       # normal demand triangular parameters
    [500, 620, 760],                       # high demand triangular parameters
]

results_df = simulationfunction(controls, inputs)
display(results_df)

In [ ]:
best_row = results_df.loc[results_df['Expected Profit'].idxmax()]
best_q = int(best_row['Order Quantity'])
print(f'Optimal order quantity = {best_q} dozen')
display(best_row.to_frame(name='Value'))

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(results_df['Order Quantity'], results_df['Expected Profit'], marker='o')
plt.xlabel('Order Quantity (dozen)')
plt.ylabel('Expected Profit')
plt.title('Expected Profit by Order Quantity')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(results_df['Order Quantity'], results_df['Stockout Probability'], marker='o')
plt.xlabel('Order Quantity (dozen)')
plt.ylabel('Stockout Probability')
plt.title('Stockout Probability by Order Quantity')
plt.tight_layout()
plt.show()

In [ ]:
# Detailed simulation at the best Q for histogram and summary stats
def simulate_one_q(controls, inputs, q):
    seed = controls[0]
    n = controls[1]
    c,p,r = inputs[0],inputs[1],inputs[2]
    supply_states = inputs[4]
    supply_prob = np.array(inputs[5], dtype=float)
    emergency_cost = inputs[6]
    high_demand_prob = inputs[7]
    normal_tri = inputs[8]
    high_tri = inputs[9]
    if seed:
        np.random.seed(1)

    state_idx = np.random.choice(len(supply_states), size=n, p=supply_prob)
    state_names = np.array(supply_states)[state_idx]
    hi_prob = np.array([high_demand_prob[s] for s in state_names])
    is_high = np.random.rand(n) < hi_prob

    demand = np.where(
        is_high,
        np.random.triangular(high_tri[0], high_tri[1], high_tri[2], size=n),
        np.random.triangular(normal_tri[0], normal_tri[1], normal_tri[2], size=n)
    )
    Ce = np.array([emergency_cost[s] for s in state_names])
    sales = np.minimum(demand, q)
    shortage = np.maximum(demand - q, 0)
    leftover = np.maximum(q - demand, 0)
    profit = sales*p + leftover*r - q*c - shortage*Ce

    return pd.DataFrame({'Profit':profit, 'Demand':demand, 'Leftover':leftover, 'Shortage':shortage})

opt_trials = simulate_one_q(controls, inputs, best_q)
display(summarystats(opt_trials[['Profit','Demand','Leftover','Shortage']]))

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(opt_trials['Profit'], bins=40)
plt.xlabel('Profit')
plt.ylabel('Frequency')
plt.title(f'Profit Distribution at Optimal Order Quantity Q = {best_q}')
plt.tight_layout()
plt.show()

In [ ]:
# Value of Information (EVPI)

def value_of_information(controls, inputs):
    """
    Computes:
    1. EV without information  = best expected profit from choosing one fixed Q now
    2. EV with perfect information = expected profit if you knew the realized scenario before choosing Q
    3. EVPI = difference between the two
    """
    seed = controls[0]
    n = controls[1]

    c = inputs[0]
    p = inputs[1]
    r = inputs[2]
    q_grid = inputs[3]
    supply_states = inputs[4]
    supply_prob = np.array(inputs[5], dtype=float)
    emergency_cost = inputs[6]
    high_demand_prob = inputs[7]
    normal_tri = inputs[8]
    high_tri = inputs[9]

    if seed:
        np.random.seed(1)

    # Simulate the underlying uncertain states once
    state_idx = np.random.choice(len(supply_states), size=n, p=supply_prob)
    state_names = np.array(supply_states)[state_idx]

    hi_prob = np.array([high_demand_prob[s] for s in state_names])
    is_high = np.random.rand(n) < hi_prob

    demand = np.where(
        is_high,
        np.random.triangular(high_tri[0], high_tri[1], high_tri[2], size=n),
        np.random.triangular(normal_tri[0], normal_tri[1], normal_tri[2], size=n)
    )

    Ce = np.array([emergency_cost[s] for s in state_names])

    # Profit matrix: one column per Q, one row per simulated scenario
    profit_matrix = []

    for q in q_grid:
        sales = np.minimum(demand, q)
        shortage = np.maximum(demand - q, 0)
        leftover = np.maximum(q - demand, 0)

        profit = sales * p + leftover * r - q * c - shortage * Ce
        profit_matrix.append(profit)

    profit_matrix = np.column_stack(profit_matrix)   # shape = (n, number_of_Qs)

    # EV without information
    avg_profit_by_q = profit_matrix.mean(axis=0)
    best_q_index = np.argmax(avg_profit_by_q)
    best_q_no_info = q_grid[best_q_index]
    ev_without_info = avg_profit_by_q[best_q_index]

    # EV with perfect information
    # For each simulated scenario, choose the best Q after seeing the realized uncertainty
    best_profit_each_scenario = profit_matrix.max(axis=1)
    best_q_each_scenario_idx = profit_matrix.argmax(axis=1)
    ev_with_perfect_info = best_profit_each_scenario.mean()

    # EVPI
    evpi = ev_with_perfect_info - ev_without_info

    # Summary table
    voi_summary = pd.DataFrame({
        'Metric': [
            'EV without information',
            'Best fixed order quantity without information',
            'EV with perfect information',
            'Expected Value of Perfect Information (EVPI)'
        ],
        'Value': [
            ev_without_info,
            best_q_no_info,
            ev_with_perfect_info,
            evpi
        ]
    })

    return voi_summary, q_choice_df, profit_matrix, demand, state_names


# Run VOI analysis
voi_summary, q_choice_df, profit_matrix, demand_sim, state_sim = value_of_information(controls, inputs)

display(voi_summary)

In [ ]:
ev_woi = voi_summary.loc[voi_summary['Metric'] == 'EV without information', 'Value'].iloc[0]
best_q_woi = voi_summary.loc[voi_summary['Metric'] == 'Best fixed order quantity without information', 'Value'].iloc[0]
ev_wpi = voi_summary.loc[voi_summary['Metric'] == 'EV with perfect information', 'Value'].iloc[0]
evpi = voi_summary.loc[voi_summary['Metric'] == 'Expected Value of Perfect Information (EVPI)', 'Value'].iloc[0]

print(f"Best fixed order quantity without information: {int(best_q_woi)} dozen")
print(f"EV without information: {ev_woi:,.2f}")
print(f"EV with perfect information: {ev_wpi:,.2f}")
print(f"EVPI: {evpi:,.2f}")